# 01 — Data Ingestion

**Purpose:** Fetch raw macroeconomic data from all sources and save to `data/raw/`.

## Sources
- **FRED** — US series: GDP growth, CPI, PCE, fed funds rate, 2y/10y Treasury yields, unemployment, M2
- **World Bank** — Cross-country GDP growth, inflation, current account balance (via `wbdata`)
- **IMF** — WEO forecasts, global growth outlook (via `imf-reader`)

## Outputs
- `data/raw/fred_series.parquet`
- `data/raw/worldbank.parquet`
- `data/raw/imf_weo.parquet`

## Papermill Parameters
- `run_date` — ISO date string injected by the GitHub Actions workflow

In [1]:
# Papermill parameters (injected at runtime)
run_date = None  # e.g. '2026-05-05'

In [ ]:
# Mount Google Drive for persistent storage (Colab only)
try:
    from google.colab import drive
    from pathlib import Path
    drive.mount("/content/drive")
    DRIVE_DATA = Path("/content/drive/MyDrive/macro-dashboard/data")
    DRIVE_DATA.mkdir(parents=True, exist_ok=True)
    (DRIVE_DATA / "raw").mkdir(exist_ok=True)
    (DRIVE_DATA / "processed").mkdir(exist_ok=True)
    (DRIVE_DATA / "outputs").mkdir(exist_ok=True)
    _IN_COLAB = True
    print("Drive mounted. Data will persist at:", DRIVE_DATA)
except Exception:
    _IN_COLAB = False
    print("Not in Colab — using local data/ directory.")

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "pandas", "requests", "wbdata", "imf-reader", "plotly",
    "papermill", "pyarrow", "pycountry", "fredapi", "scipy",
    "scikit-learn", "numpy", "yfinance"])
print("Packages ready.")

In [ ]:
import os
import pandas as pd
import wbdata
import imf_reader
from fredapi import Fred
from pathlib import Path
from datetime import datetime

RAW_DIR = DRIVE_DATA / "raw" if _IN_COLAB else Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Load FRED key: 1. Colab Secrets, 2. local .env
try:
    from google.colab import userdata
    os.environ.setdefault("FRED_API_KEY", userdata.get("FRED_API_KEY") or "")
except Exception:
    pass
for _env_file in [Path(".env"), Path("../.env")]:
    if _env_file.exists() and not os.environ.get("FRED_API_KEY"):
        for line in _env_file.read_text().splitlines():
            if "=" in line and not line.startswith("#"):
                k, v = line.split("=", 1)
                os.environ[k.strip()] = v.strip()

FRED_API_KEY = os.environ.get("FRED_API_KEY", "")
if not FRED_API_KEY:
    raise EnvironmentError("FRED_API_KEY not set. Add it to Colab Secrets or a local .env file.")

fred = Fred(api_key=FRED_API_KEY)
RUN_DATE = run_date or datetime.utcnow().strftime("%Y-%m-%d")
print(f"Run date : {RUN_DATE}")
print(f"RAW_DIR  : {RAW_DIR.resolve()}")

In [ ]:
# --- FRED ---
FRED_SERIES = {
    "gdp_growth":    "A191RL1Q225SBEA",  # Real GDP growth QoQ annualised
    "cpi_yoy":       "CPIAUCSL",          # CPI all items (index)
    "pce_yoy":       "PCEPI",             # PCE price index
    "fed_funds":     "FEDFUNDS",          # Fed funds effective rate
    "t2y":           "DGS2",              # 2-year Treasury yield
    "t10y":          "DGS10",             # 10-year Treasury yield
    "t3m":           "DGS3MO",            # 3-month Treasury yield
    "unrate":        "UNRATE",            # Unemployment rate
    "m2":            "M2SL",              # M2 money supply
    "credit_spread": "BAMLH0A0HYM2",      # ICE BofA HY OAS (credit spread)
}

frames = {}
for name, series_id in FRED_SERIES.items():
    try:
        s = fred.get_series(series_id)
        s.name = name
        frames[name] = s
        print(f"  {name}: {len(s)} obs, latest {s.index[-1].date()} = {s.iloc[-1]:.2f}")
    except Exception as e:
        print(f"  {name}: FAILED — {e}")

fred_df = pd.DataFrame(frames)
fred_df.index.name = "date"
fred_df.index = pd.to_datetime(fred_df.index)
fred_df = fred_df.sort_index()

out = RAW_DIR / "fred_series.parquet"
fred_df.to_parquet(out)
print(f"\nFRED saved: {fred_df.shape} → {out}")

In [3]:
# --- World Bank ---
WB_INDICATORS = {
    "NY.GDP.MKTP.KD.ZG": "gdp_growth",
    "FP.CPI.TOTL.ZG":    "cpi_inflation",
    "BN.CAB.XOKA.GD.ZS": "current_account_pct_gdp",
    "GC.DOD.TOTL.GD.ZS": "govt_debt_pct_gdp",
    "SL.UEM.TOTL.ZS":    "unemployment_rate",
}

# G20 economies (ISO2 codes)
COUNTRIES = [
    "US", "CN", "DE", "JP", "GB", "FR", "IN", "BR", "CA", "AU",
    "KR", "MX", "ID", "TR", "SA", "ZA", "AR", "IT", "RU", "EU",
]

print("Fetching World Bank data...")
try:
    wb_df = wbdata.get_dataframe(WB_INDICATORS, country=COUNTRIES)
    wb_df = wb_df.rename(columns=WB_INDICATORS)
    wb_df.index.names = ["country", "date"]
    wb_df = wb_df.sort_index()

    out = RAW_DIR / "worldbank.parquet"
    wb_df.to_parquet(out)
    print(f"World Bank saved: {wb_df.shape} → {out}")
    print(wb_df.tail(5))
except Exception as e:
    print(f"World Bank fetch failed: {e}")

Fetching World Bank data...
World Bank saved: (1320, 5) → data\raw\worldbank.parquet
                    gdp_growth  cpi_inflation  current_account_pct_gdp  \
country       date                                                       
United States 2021    6.055053       4.697859                -3.682741   
              2022    2.512375       8.002800                -3.878726   
              2023    2.887556       4.116338                -3.400305   
              2024    2.793001       2.949525                -4.122625   
              2025         NaN            NaN                      NaN   

                    govt_debt_pct_gdp  unemployment_rate  
country       date                                        
United States 2021         120.367733              5.349  
              2022         114.694842              3.650  
              2023         116.918809              3.638  
              2024         117.973235              4.022  
              2025                NaN     

In [ ]:
# --- IMF World Economic Outlook ---
print("Fetching IMF WEO data...")
import socket
from imf_reader import weo
from pathlib import Path

WEO_INDICATORS = [
    "NGDP_RPCH",   # Real GDP growth
    "PCPIPCH",     # CPI inflation
    "LUR",         # Unemployment rate
    "BCA_NGDPD",   # Current account % GDP
    "GGXWDG_NGDP", # Govt gross debt % GDP
]
# IMF WEO REF_AREA_LABEL strings (verified against the live WEO dataset — several
# differ from the scoreboard display names, incl. China; 02_transform renames them).
WEO_AREAS = [
    # Major basket
    "United States", "China, People's Republic of", "Germany", "Japan",
    "United Kingdom", "France", "India", "Brazil", "Canada", "Australia",
    # Emerging Markets basket (specs/006 Phase 4)
    "Mexico", "Indonesia", "Türkiye, Republic of", "South Africa",
    "Poland, Republic of", "Thailand", "Saudi Arabia", "Taiwan Province of China",
    # Developing / Frontier basket (specs/006 Phase 5)
    "Nigeria", "Vietnam", "Bangladesh", "Kenya", "Egypt, Arab Republic of", "Pakistan",
]

# Committed fallback snapshot (data/fallback/ is not gitignored). IMF's
# api.imf.org has started 401-ing (and sometimes hanging on) some datacenter/CI
# IPs, which breaks the live fetch on GitHub Actions even though it works from
# other networks. The 120s socket timeout above turns a hang into an exception.
# WEO only updates twice a year (April/October), so on any live-fetch failure we
# fall back to the committed snapshot to keep the weekly pipeline green — the
# specs/006 as-of / staleness display honestly surfaces the data aging.
FALLBACK = Path("data/fallback/imf_weo.parquet")


def _filter(df):
    return df[df["CONCEPT_CODE"].isin(WEO_INDICATORS) &
              df["REF_AREA_LABEL"].isin(WEO_AREAS)].copy()


try:
    _old_timeout = socket.getdefaulttimeout()
    socket.setdefaulttimeout(120)  # bound a hung IMF connection -> falls back below
    try:
        weo_df = _filter(weo.fetch_data())
    finally:
        socket.setdefaulttimeout(_old_timeout)
    if weo_df.empty:
        raise ValueError("IMF WEO returned no rows for the requested areas/indicators")
    print(f"IMF WEO fetched live: {weo_df.shape}")
except Exception as e:
    print(f"WARNING: IMF WEO live fetch failed ({type(e).__name__}: {e}).")
    if FALLBACK.exists():
        weo_df = _filter(pd.read_parquet(FALLBACK))
        print(f"  Using committed fallback snapshot: {weo_df.shape} <- {FALLBACK}")
    else:
        raise

out = RAW_DIR / "imf_weo.parquet"
weo_df.to_parquet(out)
print(f"IMF WEO saved: {weo_df.shape} → {out}")
print(weo_df[["CONCEPT_CODE", "REF_AREA_LABEL", "TIME_PERIOD", "OBS_VALUE"]].tail(10))

In [ ]:
# --- OECD quarterly/monthly data (primary for 9 OECD member countries) ---
# GDP quarterly YoY%, CPI monthly YoY%, unemployment monthly %
# ~1 quarter lag vs World Bank 12-24 month lag
# Countries: USA, DEU, JPN, GBR, FRA, ITA, CAN, AUS, KOR
# China, India, Brazil: not full OECD members; keep World Bank / IMF for those.

import requests, io

OECD_MEMBERS = {
    "USA": "United States",
    "DEU": "Germany",
    "JPN": "Japan",
    "GBR": "United Kingdom",
    "FRA": "France",
    "ITA": "Italy",
    "CAN": "Canada",
    "AUS": "Australia",
    "KOR": "South Korea",
}
_CODES = "+".join(OECD_MEMBERS.keys())
_BASE  = "https://sdmx.oecd.org/public/rest/data"

def _oecd_csv(url):
    # Fetch OECD SDMX CSV; return tidy DataFrame with cols: code, period, value
    r = requests.get(url, timeout=45, headers={'Accept': 'text/csv'})
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text))
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    area_col = next((c for c in df.columns if 'ref_area' in c or 'reference_area' in c), None)
    time_col = next((c for c in df.columns if 'time_period' in c), None)
    val_col  = next((c for c in df.columns if 'obs_value' in c or 'observation_value' in c), None)
    if not all([area_col, time_col, val_col]):
        raise ValueError(f'Unexpected CSV columns: {df.columns.tolist()}')
    return df[[area_col, time_col, val_col]].rename(
        columns={area_col: 'code', time_col: 'period', val_col: 'value'}
    ).dropna(subset=['value'])

print('Fetching OECD data for 9 member countries...')
oecd_frames = []

# 1. GDP growth — quarterly YoY %
try:
    gdp_url = (
        f'{_BASE}/OECD.SDD.NAD,DSD_NAMAIN1@DF_QNA_EXPENDITURE_NATIONAL,1.0/'
        f'Q.{_CODES}.B1GQ..YA_GY'
        f'?startPeriod=2020-Q1&format=csv'
    )
    gdp = _oecd_csv(gdp_url)
    gdp['indicator'] = 'oecd_gdp_yoy'
    def _q_to_date(p):
        yr, q = p.split('-Q')
        month = int(q) * 3
        return pd.Timestamp(year=int(yr), month=month, day=1) + pd.offsets.MonthEnd(0)
    gdp['date'] = gdp['period'].apply(_q_to_date)
    oecd_frames.append(gdp[['code', 'date', 'indicator', 'value']])
    print(f'  GDP:          {len(gdp)} rows, latest {gdp["period"].max()}')
except Exception as e:
    print(f'  GDP fetch failed: {e}')

# 2. CPI inflation — monthly YoY %
try:
    cpi_url = (
        f'{_BASE}/OECD.SDD.TPS,DSD_PRICES@DF_PRICES_T,1.0/'
        f'M.{_CODES}.CPI.PA.._T.N.GY'
        f'?startPeriod=2020-01&format=csv'
    )
    cpi = _oecd_csv(cpi_url)
    cpi['indicator'] = 'oecd_cpi_yoy'
    cpi['date'] = pd.to_datetime(cpi['period']) + pd.offsets.MonthEnd(0)
    oecd_frames.append(cpi[['code', 'date', 'indicator', 'value']])
    print(f'  CPI:          {len(cpi)} rows, latest {cpi["period"].max()}')
except Exception as e:
    print(f'  CPI fetch failed: {e}')

# 3. Unemployment — monthly % of labour force
try:
    unemp_url = (
        f'{_BASE}/OECD.SDD.TPS,DSD_LFS@DF_IALFS_INDIC,1.0/'
        f'M.{_CODES}..._T..UNE_LF.PT_LF_TOTEM'
        f'?startPeriod=2020-01&format=csv'
    )
    unemp = _oecd_csv(unemp_url)
    unemp['indicator'] = 'oecd_unemployment'
    unemp['date'] = pd.to_datetime(unemp['period']) + pd.offsets.MonthEnd(0)
    oecd_frames.append(unemp[['code', 'date', 'indicator', 'value']])
    print(f'  Unemployment: {len(unemp)} rows, latest {unemp["period"].max()}')
except Exception as e:
    print(f'  Unemployment fetch failed: {e}')

if oecd_frames:
    oecd_long = pd.concat(oecd_frames, ignore_index=True)
    oecd_long['country'] = oecd_long['code'].map(OECD_MEMBERS)
    oecd_wide = oecd_long.pivot_table(
        index=['country', 'date'], columns='indicator', values='value', aggfunc='last'
    ).rename_axis(columns=None).sort_index()
    out = RAW_DIR / 'oecd_country.parquet'
    oecd_wide.to_parquet(out)
    print(f'OECD saved: {oecd_wide.shape} -> {out}')
    us = oecd_wide.xs('United States', level='country') if 'United States' in oecd_wide.index.get_level_values('country') else oecd_wide
    print(us.tail(4))
else:
    print('WARNING: No OECD data fetched — oecd_country.parquet not created')


In [ ]:
# --- Stock indices YTD performance (yfinance) ---
import yfinance as yf
from datetime import datetime

STOCK_INDICES = {
    "United States":  "^GSPC",
    "Germany":        "^GDAXI",
    "Japan":          "^N225",
    "United Kingdom": "^FTSE",
    "France":         "^FCHI",
    "China":          "000001.SS",
    "India":          "^BSESN",
    "Brazil":         "^BVSP",
    "Canada":         "^GSPTSE",
    "Australia":      "^AXJO",
    "South Korea":    "^KS11",
    "Italy":          "FTSEMIB.MI",
}

year_start = f"{datetime.utcnow().year}-01-01"
rows = []
for country, ticker in STOCK_INDICES.items():
    try:
        hist = yf.download(ticker, start=year_start, progress=False, auto_adjust=True)
        # yfinance >=0.2.x may return MultiIndex columns for single tickers
        close_col = hist["Close"]
        if isinstance(close_col, pd.DataFrame):
            close_col = close_col.iloc[:, 0]
        closes = close_col.dropna()
        if len(closes) >= 2:
            ytd = round((float(closes.iloc[-1]) / float(closes.iloc[0]) - 1) * 100, 2)
            rows.append({"country": country, "stock_ytd_pct": ytd,
                         "stock_ytd_as_of": str(closes.index[-1].date())})
            print(f"  {country}: {ytd:+.2f}%  ({ticker})")
        else:
            print(f"  {country}: insufficient data")
    except Exception as e:
        print(f"  {country}: FAILED — {e}")

if rows:
    stocks_df = pd.DataFrame(rows).set_index("country")[["stock_ytd_pct", "stock_ytd_as_of"]]
else:
    print("WARNING: No stock data fetched — saving empty file.")
    stocks_df = pd.DataFrame(columns=["stock_ytd_pct", "stock_ytd_as_of"])
    stocks_df.index.name = "country"
out = RAW_DIR / "stock_indices.parquet"
stocks_df.to_parquet(out)
print(f"\nStock indices saved: {stocks_df.shape} → {out}")

In [ ]:
# --- Central bank policy rates (FRED) ---
POLICY_RATE_SERIES = {
    "United States":  "FEDFUNDS",
    "Germany":        "ECBDFR",
    "France":         "ECBDFR",
    "Italy":          "ECBDFR",
    "United Kingdom": "BOERUKM",
    "Japan":          "IRSTCI01JPM156N",
    "Canada":         "IRSTCI01CAM156N",
    "Australia":      "IRSTCI01AUM156N",
    "India":          "IRSTCI01INM156N",
    "Brazil":         "IRSTCI01BRM156N",
    "South Korea":    "IRSTCI01KRM156N",
    "China":          "IRSTCI01CNM156N",
}

policy_rows = []
for country, series_id in POLICY_RATE_SERIES.items():
    try:
        s = fred.get_series(series_id).dropna()
        rate = round(float(s.iloc[-1]), 2)
        policy_rows.append({"country": country, "policy_rate": rate,
                            "policy_rate_as_of": str(s.index[-1].date())})
        print(f"  {country}: {rate:.2f}%")
    except Exception as e:
        print(f"  {country}: FAILED — {e}")

policy_df = pd.DataFrame(policy_rows).set_index("country")
out = RAW_DIR / "policy_rates.parquet"
policy_df.to_parquet(out)
print(f"\nPolicy rates saved: {policy_df.shape} → {out}")